# Episode 3 — Bootstrapping an AUD AONIA curve with QuantLib

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **Illustrative data.** The quotes in `quotes_illustrative.csv` are made up for teaching. They are shaped like a plausible AUD OIS curve but are **not market prices**. Educational material only, not investment advice.

1. Setup · 2. Conventions · 3. Calendar · 4. Quotes · 5. Bootstrap · 6. Hand check · 7. Cash flows · 8. Reprice check · 9. Interpolation · 10. First DV01 · 11. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data file this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `quotes_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_illustrative.csv').write_text("""tenor,instrument,rate_pct,note
O/N,deposit,3.85,Stand-in for today's AONIA (not published until tomorrow): modelling choice
1M,ois,3.86,
3M,ois,3.89,
6M,ois,3.93,
9M,ois,3.97,
1Y,ois,4.00,
18M,ois,4.05,
2Y,ois,4.08,
3Y,ois,4.13,
5Y,ois,4.24,
7Y,ois,4.35,
10Y,ois,4.50,
""")
print('wrote quotes_illustrative.csv')

In [ ]:
# Parameters (papermill overrides these)
valuation_date = "2026-09-22"
quotes_file = "quotes_illustrative.csv"
output_json = "build/outputs.json"
notional = 100_000_000
dv01_tenor = "5Y"

## 1. Setup

In [ ]:
import json
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import QuantLib as ql

today = ql.DateParser.parseISO(valuation_date)
ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
iso = lambda d: d.ISO()

out = {"meta": {
    "episode": 3,
    "valuation_date": valuation_date,
    "quantlib_version": ql.__version__,
    "quotes_label": "ILLUSTRATIVE - not market data",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
print("QuantLib", ql.__version__, "| valuation date", today)

## 2. Conventions

AUD AONIA OIS conventions used below. Sources: AFMA *Interest Rate Derivative Conventions* (June 2025) and the RBA *Cash Rate Methodology* page.

| Item | Convention | Source |
|---|---|---|
| Floating rate | RBA Interbank Overnight Cash Rate, also known as AONIA | RBA Cash Rate Methodology; AFMA §2.2 |
| Day count | ACT/365 Fixed on both legs; no leap-year adjustment | AFMA §3.7, Glossary |
| Start date | next Sydney business day after the trade (T+1) | AFMA §5.2, §3.3 example |
| Business days | not a NSW bank close day; Modified Following; no end-of-month rule for single-currency AUD swaps | AFMA §3.3 |
| Payment frequency | ≤ 12 months: once at maturity. > 12 months: front stub (if any), then annually | AFMA §3.7, §5.2 |
| Payment lag | two business days after each period ends | AFMA §3.17, §5.2 |
| Compounding | daily; weekends and holidays add their days to the previous business day | AFMA §3.17 |
| Publication | the rate for day *d* is published the next business day before 9:20am | RBA Cash Rate Methodology |
| Fixings | the RBA's published series, including NSW-only holidays it publishes on | RBA Statistical Table F1 |

Where AFMA and the RBA differ on AONIA itself (publication time, which days have a rate), the RBA, as administrator, is the source of truth.

In [ ]:
CONV = {
    "settlement_days": 1,
    "payment_lag_days": 2,
    "payment_frequency": "Annual (single payment if <= 12M)",
    "business_day_convention": "Modified Following",
    "end_of_month": False,
    "day_count": dc.name(),
    "compounding": "daily, in arrears",
}
out["conventions"] = CONV
pd.Series(CONV)

## 3. Calendar

AFMA defines a business day as any day that is not a bank close day under New South Wales law. QuantLib's `Australia` settlement calendar is close, but it is missing the NSW *additional days* for Anzac Day when 25 April falls on a weekend. We add them by hand.

`ql.Aonia()` builds its own copy of the calendar, so a holiday added here does **not** reach it. We therefore define the AONIA index ourselves, on the corrected calendar.

**Fixing days are not quite Sydney business days.** RBA Statistical Table F1 shows AONIA published on some NSW-only holidays (for example 27 April 2026 and the 3 August 2026 Bank Holiday), because interbank settlement was open. The RBA's published series is the source of truth for fixings, while swap dates and payments still roll on Sydney business days. Section 6b shows why this has no effect on a curve bootstrapped from quotes.

In [ ]:
cal = ql.Australia(ql.Australia.Settlement)
NSW_ADDITIONAL_DAYS = [ql.Date(27, 4, 2026), ql.Date(26, 4, 2027)]  # NSW Government public holiday list

missing = [d for d in NSW_ADDITIONAL_DAYS if cal.isBusinessDay(d)]
for d in missing:
    cal.addHoliday(d)

aonia = ql.OvernightIndex("AONIA", 0, ql.AUDCurrency(), cal, dc)

check = pd.DataFrame({
    "date": [iso(d) for d in NSW_ADDITIONAL_DAYS],
    "missing in QuantLib": [d in missing for d in NSW_ADDITIONAL_DAYS],
    "ql.Aonia() business day?": [ql.Aonia().fixingCalendar().isBusinessDay(d) for d in NSW_ADDITIONAL_DAYS],
    "our AONIA business day?": [aonia.fixingCalendar().isBusinessDay(d) for d in NSW_ADDITIONAL_DAYS],
})
spot = cal.advance(today, CONV["settlement_days"], ql.Days)
out["calendar"] = {"added_holidays": [iso(d) for d in missing], "spot_date": iso(spot),
                   "holidays_2026_2027": [iso(d) for d in cal.holidayList(ql.Date(1, 1, 2026), ql.Date(31, 12, 2027))]}
print("spot date:", spot)
check

## 4. Quotes (illustrative)

In [ ]:
quotes = pd.read_csv(quotes_file).fillna("")
out["quotes"] = quotes[["tenor", "instrument", "rate_pct"]].to_dict("records")
quotes

## 5. Bootstrap

For an OIS of up to 12 months there is a single period from start $s$ to end $e$, paid at $p$. When the same curve both forecasts AONIA and discounts:

$$\text{PV}_\text{float} = \Big(\tfrac{P(s)}{P(e)} - 1\Big) P(p), \qquad \text{PV}_\text{fixed} = S\,\tau\,P(p)$$

Setting them equal, the payment-date factor $P(p)$ cancels:

$$P(e) = \frac{P(s)}{1 + S\,\tau}, \qquad \tau = \frac{\text{days}(s,e)}{365}$$

For longer swaps with no payment lag, the floating leg telescopes to $P(t_0) - P(t_n)$, giving $S = \dfrac{P(t_0) - P(t_n)}{\sum_i \tau_i P(t_i)}$, which we solve for the last unknown $P(t_n)$, one pillar at a time. With the two-day payment lag, QuantLib solves the exact condition numerically.

Interpolation: log-linear on discount factors, which means piecewise-flat instantaneous forwards.

In [ ]:
def make_helpers():
    # Fresh helpers and quotes each call: a helper can only belong to one curve.
    qs, hs = {}, []
    for row in quotes.itertuples():
        q = ql.SimpleQuote(row.rate_pct / 100)
        qs[row.tenor] = q
        if row.instrument == "deposit":
            h = ql.DepositRateHelper(ql.QuoteHandle(q), ql.Period(1, ql.Days), 0, cal,
                                     ql.Following, False, dc)
        else:
            h = ql.OISRateHelper(CONV["settlement_days"], ql.Period(row.tenor), ql.QuoteHandle(q), aonia,
                                 paymentLag=CONV["payment_lag_days"], paymentFrequency=ql.Annual,
                                 paymentCalendar=cal, convention=ql.ModifiedFollowing, endOfMonth=False)
        hs.append((row.tenor, row.instrument, h))
    return qs, hs

quote_handles, helpers = make_helpers()
curve = ql.PiecewiseLogLinearDiscount(today, [h for *_, h in helpers], dc)
curve.enableExtrapolation()
curve_handle = ql.YieldTermStructureHandle(curve)

pillars = pd.DataFrame([{
    "tenor": tenor,
    "instrument": kind,
    "quote_pct": quote_handles[tenor].value() * 100,
    "maturity": iso(h.maturityDate()),
    "pillar_date": iso(h.pillarDate()),
    "t_years": dc.yearFraction(today, h.pillarDate()),
    "discount_factor": curve.discount(h.pillarDate()),
    "zero_cc_pct": curve.zeroRate(h.pillarDate(), dc, ql.Continuous).rate() * 100,
} for tenor, kind, h in helpers])
out["pillars"] = pillars.to_dict("records")
pillars.style.format({"quote_pct": "{:.2f}", "t_years": "{:.4f}", "discount_factor": "{:.8f}", "zero_cc_pct": "{:.4f}"})

## 6. Hand check: the first two pillars

The overnight pillar comes from today's rate over the nights to spot. Then the 1M OIS gives $P(e) = P(s)/(1 + S\tau)$. If the hand calculation and QuantLib disagree, the conventions in the code are wrong.

In [ ]:
r_on = quote_handles["O/N"].value()
on_days = spot - today
df_spot_hand = 1 / (1 + r_on * on_days / 365)

h1m = dict((t, h) for t, _, h in helpers)["1M"]
s, e = h1m.swap().startDate(), h1m.swap().maturityDate()
S_1m = quote_handles["1M"].value()
days_1m = e - s
tau_1m = days_1m / 365
df_1m_hand = df_spot_hand / (1 + S_1m * tau_1m)

hand = {
    "on_rate_pct": r_on * 100, "on_days": on_days, "spot_date": iso(spot),
    "df_spot_hand": df_spot_hand, "df_spot_ql": curve.discount(spot),
    "ois_1m_rate_pct": S_1m * 100, "ois_1m_start": iso(s), "ois_1m_end": iso(e),
    "ois_1m_days": days_1m, "ois_1m_tau": tau_1m,
    "df_1m_hand": df_1m_hand, "df_1m_ql": curve.discount(e),
}
hand["abs_diff_spot"] = abs(hand["df_spot_hand"] - hand["df_spot_ql"])
hand["abs_diff_1m"] = abs(hand["df_1m_hand"] - hand["df_1m_ql"])
out["hand_check"] = hand
pd.Series(hand)

## 6b. Does the fixing calendar change the curve?

Compound projected AONIA over the 1Y swap period two ways: fixing on Sydney business days only, and also fixing on the NSW-only holidays the RBA has historically published on. Each daily factor is $P(d_i)/P(d_{i+1})$, so the product telescopes to $P(s)/P(e)$ however the days are split. The choice only matters for **historical** fixings (Episode 7).

In [ ]:
h1y = dict((t, h) for t, _, h in helpers)["1Y"]
s1, e1 = h1y.swap().startDate(), h1y.swap().maturityDate()
# NSW-only holidays in the period. F1 shows the RBA published on the 2025/2026 equivalents; assumed to continue.
NSW_ONLY = [ql.Date(5, 10, 2026), ql.Date(26, 4, 2027), ql.Date(2, 8, 2027)]
extra = [d for d in NSW_ONLY if s1 <= d < e1 and not cal.isBusinessDay(d)]

def compounded(fixing_days):
    acc = 1.0
    for d0, d1 in zip(fixing_days, fixing_days[1:] + [e1]):
        acc *= 1 + curve.forwardRate(d0, d1, dc, ql.Simple).rate() * (d1 - d0) / 365
    return acc - 1

sydney_days, d = [], s1
while d < e1:
    if cal.isBusinessDay(d):
        sydney_days.append(d)
    d += 1
rba_days = sorted(set(sydney_days) | set(extra))
fix = {
    "period_start": iso(s1), "period_end": iso(e1),
    "fixings_sydney": len(sydney_days), "fixings_rba": len(rba_days), "extra_fixing_days": [iso(x) for x in extra],
    "compounded_sydney": compounded(sydney_days), "compounded_rba": compounded(rba_days),
    "telescoped": curve.discount(s1) / curve.discount(e1) - 1,
}
fix["abs_diff"] = abs(fix["compounded_sydney"] - fix["compounded_rba"])
out["fixing_days_check"] = fix
pd.Series(fix)

## 7. Cash flows of an 18-month OIS

Longer than 12 months, so AFMA's rule applies: a front stub, then annual payments, each paid two business days after the period ends. Quoted at par, the two legs have equal value.

In [ ]:
aonia_fwd = aonia.clone(curve_handle)

def make_ois(tenor, rate, receive_fixed=True):
    return ql.MakeOIS(ql.Period(tenor), aonia_fwd, rate, ql.Period(0, ql.Days),
                      settlementDays=CONV["settlement_days"], paymentLag=CONV["payment_lag_days"],
                      paymentFrequency=ql.Annual, paymentCalendar=cal, discountingTermStructure=curve_handle,
                      nominal=notional, receiveFixed=receive_fixed, endOfMonth=False)

ois18 = make_ois("18M", quote_handles["18M"].value())
rows = []
for f, o in zip(ois18.fixedLeg(), ois18.overnightLeg()):
    fc, oc = ql.as_coupon(f), ql.as_coupon(o)
    df = curve.discount(f.date())
    rows.append({
        "accrual_start": iso(fc.accrualStartDate()), "accrual_end": iso(fc.accrualEndDate()),
        "payment_date": iso(f.date()), "days": fc.accrualDays(), "tau": fc.accrualPeriod(),
        "fixed_amount": f.amount(), "float_amount": o.amount(),
        "float_rate_pct": ql.as_floating_rate_coupon(o).rate() * 100,
        "df_payment": df, "pv_fixed": f.amount() * df, "pv_float": o.amount() * df,
    })
cf18 = pd.DataFrame(rows)
out["cashflows_18m"] = {"notional": notional, "fixed_rate_pct": quote_handles["18M"].value() * 100,
                        "rows": rows, "npv": ois18.NPV()}
print("NPV at par:", round(ois18.NPV(), 6))
cf18

## 8. Reprice check

A bootstrapped curve must reprice every input instrument to its quote.

In [ ]:
rep = pd.DataFrame([{
    "tenor": t, "quote_pct": quote_handles[t].value() * 100,
    "implied_pct": h.impliedQuote() * 100,
    "error_bp": (h.impliedQuote() - quote_handles[t].value()) * 1e4,
} for t, _, h in helpers])
out["reprice"] = {"rows": rep.to_dict("records"), "max_abs_error_bp": float(rep.error_bp.abs().max())}
print("max |error| (bp):", out["reprice"]["max_abs_error_bp"])
rep

## 9. Interpolation changes the forwards, not the fit

We rebuild the curve with a natural cubic spline on zero rates. Both curves reprice every quote exactly. What differs is how they fill in between the pillars.

Flat forwards step at *our* pillar dates. Real policy steps happen at RBA Monetary Policy Board meetings, which is why desks also quote meeting-dated OIS (a later episode).

In [ ]:
_, helpers_cubic = make_helpers()
curve_cubic = ql.PiecewiseNaturalCubicZero(today, [h for *_, h in helpers_cubic], dc)
curve_cubic.enableExtrapolation()

grid = []
for m in range(0, 120):
    d0 = cal.advance(spot, m, ql.Months)
    d1 = cal.advance(spot, m + 1, ql.Months)
    grid.append({
        "date": iso(d0), "t_years": dc.yearFraction(today, d0),
        "zero_flat_pct": curve.zeroRate(d0, dc, ql.Continuous).rate() * 100,
        "zero_cubic_pct": curve_cubic.zeroRate(d0, dc, ql.Continuous).rate() * 100,
        "fwd1m_flat_pct": curve.forwardRate(d0, d1, dc, ql.Simple).rate() * 100,
        "fwd1m_cubic_pct": curve_cubic.forwardRate(d0, d1, dc, ql.Simple).rate() * 100,
    })
grid = pd.DataFrame(grid)
cubic_err = max(abs(h.impliedQuote() - quote_handles[t].value()) * 1e4 for t, _, h in helpers_cubic)

# RBA Monetary Policy Board: decision on the second day of each two-day meeting (RBA schedule page)
RBA_DECISIONS = ["2026-09-29", "2026-11-03", "2026-12-08", "2027-02-09", "2027-03-23", "2027-05-04",
                 "2027-06-22", "2027-08-10", "2027-09-28", "2027-11-02", "2027-12-14"]
out["curves"] = {"grid": grid.to_dict("records"), "cubic_max_abs_error_bp": cubic_err,
                 "interpolations": {"flat": "log-linear discount (piecewise flat forward)",
                                    "cubic": "natural cubic spline on zero rates"}}
out["rba_decisions"] = RBA_DECISIONS

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(grid.t_years, grid.zero_flat_pct, label="flat forward")
ax[0].plot(grid.t_years, grid.zero_cubic_pct, "--", label="cubic zero")
ax[0].scatter(pillars.t_years, pillars.zero_cc_pct, s=18, color="k", zorder=3, label="pillars")
ax[0].set(title="Zero rate (cont., ACT/365F)", xlabel="years", ylabel="%")
ax[1].step(grid.t_years, grid.fwd1m_flat_pct, where="post", label="flat forward")
ax[1].plot(grid.t_years, grid.fwd1m_cubic_pct, "--", label="cubic zero")
ax[1].set(title="1M forward rate (simple, ACT/365F)", xlabel="years", ylabel="%")
for a in ax:
    a.grid(alpha=.3); a.legend()
plt.tight_layout()
print("cubic curve max |reprice error| (bp):", cubic_err)

## 10. First DV01

Receive fixed on a 5Y AONIA OIS at the par rate, AUD 100m notional. Shift **every quote** by ±1bp, re-bootstrap, reprice. This is a *par* DV01: sensitivity to the quotes, not to zero rates (Episode 8).

AFMA's customary OIS parcel is sized to about AUD 25,000 per basis point (§3.4), so we also back out the matching notional.

In [ ]:
par = quote_handles[dv01_tenor].value()
swap = make_ois(dv01_tenor, par, receive_fixed=True)

def npv_after_shift(bp):
    for q in quote_handles.values():
        q.setValue(q.value() + bp / 1e4)
    v = swap.NPV()
    for q in quote_handles.values():
        q.setValue(q.value() - bp / 1e4)
    return v

base = swap.NPV()
up, down = npv_after_shift(+1), npv_after_shift(-1)
dv01 = (down - up) / 2  # value gained per 1bp fall in rates (receiver is long rates-down)
out["dv01"] = {
    "tenor": dv01_tenor, "notional": notional, "par_rate_pct": par * 100, "side": "receive fixed",
    "npv_base": base, "npv_up_1bp": up, "npv_down_1bp": down, "dv01": dv01,
    "dv01_per_million": dv01 / (notional / 1e6),
    "notional_for_25k_per_bp": 25_000 / (dv01 / notional),
}
pd.Series(out["dv01"])

## 11. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")